# PASO 1. Descargar nuestra base de datos desde MT5

En este notebook descargaremos desde MetaTrader 5:

- Las 99.999 velas M1 cerradas más recientes de EUR/USD.
- Los ticks Bid/Ask de los últimos 7 días.

Esta descarga constituirá nuestra base histórica inicial.

Los datos se guardarán en `data/raw/` con la fecha de la descarga
para conservarlos como una fotografía fija y evitar sobrescribirlos
durante el desarrollo de la estrategia.

### Importaciones y configuración

In [1]:
from pathlib import Path
from datetime import datetime, timezone, timedelta

import MetaTrader5 as mt5
import pandas as pd


# --------------------------------------------------
# CONFIGURACIÓN GENERAL
# --------------------------------------------------
SYMBOL = "EURUSD"

# Número de velas M1 cerradas
N_BARS = 99_999

# Número de días de ticks Bid/Ask
TICKS_DAYS = 7

# Carpeta de datos originales
DATA_RAW = Path("../data/raw")

DATA_RAW.mkdir(
    parents=True,
    exist_ok=True
)

print("Carpeta de datos:")
print(DATA_RAW.resolve())

Carpeta de datos:
C:\ALL\TECNICO\TRADING\CURSOS\TUTORIAS\PROYECTO2\mt5_microstructure_lab\data\raw


## 1. Conectar Python con MetaTrader 5

Comprobamos que Python puede comunicarse correctamente con
MetaTrader 5 y verificamos la cuenta demo activa.

In [2]:
print(
    "Paquete MetaTrader5:",
    mt5.__version__
)

if not mt5.initialize():
    raise RuntimeError(
        f"No se pudo iniciar MT5: {mt5.last_error()}"
    )

terminal = mt5.terminal_info()
account = mt5.account_info()

if terminal is None:
    raise RuntimeError(
        "No se pudo obtener información del terminal MT5."
    )

if account is None:
    raise RuntimeError(
        "No se pudo obtener información de la cuenta."
    )

print("Terminal conectado:", terminal.connected)
print("Cuenta demo conectada correctamente.")
print("Servidor:", account.server)
print("Balance:", account.balance)
print("Moneda:", account.currency)
print("Apalancamiento:", f"1:{account.leverage}")
print("Versión terminal:", mt5.version())

Paquete MetaTrader5: 5.0.6090
Terminal conectado: True
Cuenta: 112277233
Servidor: MetaQuotes-Demo
Balance: 10000.0
Moneda: EUR
Apalancamiento: 1:33
Versión terminal: (500, 6182, '5 Sep 2026')


## 2. Seleccionar EUR/USD

Seleccionamos EUR/USD y obtenemos las características necesarias
del símbolo, especialmente el número de decimales y el tamaño
de un punto.

In [3]:
if not mt5.symbol_select(
    SYMBOL,
    True
):
    raise RuntimeError(
        f"No se pudo seleccionar {SYMBOL}: "
        f"{mt5.last_error()}"
    )


info = mt5.symbol_info(
    SYMBOL
)

if info is None:
    raise RuntimeError(
        f"No se pudo obtener información de {SYMBOL}"
    )


tick_actual = mt5.symbol_info_tick(
    SYMBOL
)

if tick_actual is None:
    raise RuntimeError(
        f"No se pudo obtener el tick actual de {SYMBOL}"
    )


POINT = info.point


print("Símbolo:", SYMBOL)
print("Descripción:", info.description)
print("Digits:", info.digits)
print("Point:", POINT)
print("Bid actual:", tick_actual.bid)
print("Ask actual:", tick_actual.ask)

Símbolo: EURUSD
Descripción: Euro vs US Dollar
Digits: 5
Point: 1e-05
Bid actual: 1.1622
Ask actual: 1.16226


In [4]:
# --------------------------------------------------
# HORA DE CORTE DE NUESTRA BASE HISTÓRICA
# --------------------------------------------------

tick_snapshot = mt5.symbol_info_tick(
    SYMBOL
)

if tick_snapshot is None:
    raise RuntimeError(
        f"No se pudo obtener el tick actual de {SYMBOL}"
    )

SNAPSHOT_TIME = pd.to_datetime(
    tick_snapshot.time,
    unit="s",
    utc=True
)

print(
    "Hora de corte MT5:",
    SNAPSHOT_TIME
)

Hora de corte MT5: 2026-09-08 00:17:00+00:00


## 3. Descargar las velas M1 cerradas más recientes

Descargamos las 99.999 velas M1 cerradas más recientes disponibles
en MetaTrader 5.

Comenzamos en la posición 1 para excluir la vela actual,
que todavía se encuentra en formación.

La posición 0 representa la vela actual y la numeración continúa hacia el pasado, según la documentación oficial de copy_rates_from_pos().

In [5]:
rates = mt5.copy_rates_from_pos(
    SYMBOL,
    mt5.TIMEFRAME_M1,
    1,          # 1 = última vela completamente cerrada
    N_BARS
)

if rates is None:
    raise RuntimeError(
        f"Error descargando velas M1: "
        f"{mt5.last_error()}"
    )

if len(rates) == 0:
    raise RuntimeError(
        "MT5 no devolvió ninguna vela M1."
    )


m1_df = pd.DataFrame(
    rates
)


m1_df["time"] = pd.to_datetime(
    m1_df["time"],
    unit="s",
    utc=True
)


m1_df = (
    m1_df
    .sort_values("time")
    .reset_index(drop=True)
)


print(
    "Velas descargadas:",
    len(m1_df)
)

print(
    "Primera vela:",
    m1_df["time"].min()
)

print(
    "Última vela:",
    m1_df["time"].max()
)

print(
    "\nColumnas:"
)

print(
    m1_df.columns.tolist()
)


display(
    m1_df.head()
)

display(
    m1_df.tail()
)

Velas descargadas: 99999
Primera vela: 2026-06-02 13:34:00+00:00
Última vela: 2026-09-08 00:16:00+00:00

Columnas:
['time', 'open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume']


,time,open,high,low,close,tick_volume,spread,real_volume
0,2026-06-02 13:34:00+00:00,1.16422,1.16427,1.16421,1.16422,28,2,0
1,2026-06-02 13:35:00+00:00,1.16421,1.16423,1.16416,1.16417,16,2,0
2,2026-06-02 13:36:00+00:00,1.16417,1.16423,1.16416,1.16421,30,2,0
3,2026-06-02 13:37:00+00:00,1.16421,1.16427,1.16421,1.16424,17,2,0
4,2026-06-02 13:38:00+00:00,1.16424,1.16426,1.16424,1.16426,11,2,0


,time,open,high,low,close,tick_volume,spread,real_volume
99994,2026-09-08 00:12:00+00:00,1.16227,1.16227,1.16219,1.16219,3,2,0
99995,2026-09-08 00:13:00+00:00,1.16219,1.16219,1.16216,1.16216,2,4,0
99996,2026-09-08 00:14:00+00:00,1.16210,1.16216,1.16210,1.16216,2,7,0
99997,2026-09-08 00:15:00+00:00,1.16216,1.16218,1.16216,1.16216,6,5,0
99998,2026-09-08 00:16:00+00:00,1.16216,1.16219,1.16216,1.16218,7,6,0


## 4. Descargar los ticks Bid/Ask

Descargamos los ticks de EUR/USD correspondientes a los últimos
7 días.

Estos datos se utilizarán posteriormente para estudiar Bid, Ask,
spread, slippage, latencia y ejecución.

In [6]:
# --------------------------------------------------
# DESCARGAR TICKS BID/ASK
# --------------------------------------------------

tick_fin = SNAPSHOT_TIME.to_pydatetime()

tick_inicio = (
    tick_fin
    - timedelta(days=TICKS_DAYS)
)


print(
    "Inicio de ticks:",
    tick_inicio
)

print(
    "Fin de ticks:",
    tick_fin
)


ticks_raw = mt5.copy_ticks_range(
    SYMBOL,
    tick_inicio,
    tick_fin,
    mt5.COPY_TICKS_INFO
)


if ticks_raw is None:
    raise RuntimeError(
        f"Error descargando ticks: "
        f"{mt5.last_error()}"
    )


if len(ticks_raw) == 0:
    raise RuntimeError(
        "MT5 no devolvió ningún tick."
    )


print(
    "Ticks descargados:",
    len(ticks_raw)
)

Inicio de ticks: 2026-09-01 00:17:00+00:00
Fin de ticks: 2026-09-08 00:17:00+00:00
Ticks descargados: 1064413


## 5. Preparar los ticks y calcular el spread

Convertimos los ticks a un DataFrame de Pandas.

Calculamos:

Spread = Ask - Bid

y expresamos también el spread en puntos.

In [7]:
ticks_df = pd.DataFrame(
    ticks_raw
)


ticks_df["time"] = pd.to_datetime(
    ticks_df["time_msc"],
    unit="ms",
    utc=True
)


ticks_df = (
    ticks_df
    .sort_values("time")
    .reset_index(drop=True)
)


ticks_df["spread_price"] = (
    ticks_df["ask"]
    - ticks_df["bid"]
)


ticks_df["spread_points"] = (
    ticks_df["spread_price"]
    / POINT
)


print(
    "Número de ticks:",
    len(ticks_df)
)

print(
    "Primera fecha:",
    ticks_df["time"].min()
)

print(
    "Última fecha:",
    ticks_df["time"].max()
)


display(
    ticks_df[
        [
            "time",
            "bid",
            "ask",
            "spread_price",
            "spread_points"
        ]
    ].head(20)
)

Número de ticks: 1064413
Primera fecha: 2026-09-01 00:17:05.574000+00:00
Última fecha: 2026-09-08 00:16:56.458000+00:00


,time,bid,ask,spread_price,spread_points
0,2026-09-01 00:17:05.574000+00:00,1.16164,1.16173,0.00009,9.0
1,2026-09-01 00:17:05.775000+00:00,1.16164,1.16172,0.00008,8.0
2,2026-09-01 00:17:06.175000+00:00,1.16164,1.16173,0.00009,9.0
3,2026-09-01 00:17:06.276000+00:00,1.16164,1.16172,0.00008,8.0
4,2026-09-01 00:17:06.527000+00:00,1.16164,1.16173,0.00009,9.0
5,2026-09-01 00:17:06.576000+00:00,1.16164,1.16172,0.00008,8.0
6,2026-09-01 00:17:49.295000+00:00,1.16164,1.16174,0.00010,10.0
7,2026-09-01 00:17:49.328000+00:00,1.16166,1.16173,0.00007,7.0
8,2026-09-01 00:17:49.420000+00:00,1.16168,1.16173,0.00005,5.0
9,2026-09-01 00:17:51.269000+00:00,1.16168,1.16174,0.00006,6.0


## 6. Comprobaciones básicas

Antes de guardar nuestra base original realizamos unas comprobaciones
mínimas.

No eliminaremos ni modificaremos datos en este notebook.

El análisis completo de calidad corresponde a `02_data_quality.ipynb`.

In [8]:
print(
    "========== VELAS M1 =========="
)

print(
    "Nulos totales:",
    m1_df.isna().sum().sum()
)

print(
    "Fechas duplicadas:",
    m1_df["time"].duplicated().sum()
)

print(
    "Orden cronológico:",
    m1_df["time"].is_monotonic_increasing
)


print(
    "\n========== TICKS =========="
)

print(
    "Bid <= 0:",
    (ticks_df["bid"] <= 0).sum()
)

print(
    "Ask <= 0:",
    (ticks_df["ask"] <= 0).sum()
)

print(
    "Ask < Bid:",
    (ticks_df["ask"] < ticks_df["bid"]).sum()
)

print(
    "Spread negativo:",
    (
        ticks_df["spread_points"] < 0
    ).sum()
)

print(
    "Registros con misma marca temporal:",
    ticks_df["time"].duplicated().sum()
)

print(
    "Orden cronológico:",
    ticks_df["time"].is_monotonic_increasing
)
# Las marcas temporales repetidas se estudiarán
# en 02_data_quality.ipynb.

========== VELAS M1 ==========
Nulos totales: 0
Fechas duplicadas: 0
Orden cronológico: True

========== TICKS ==========
Bid <= 0: 0
Ask <= 0: 0
Ask < Bid: 0
Spread negativo: 0
Registros con misma marca temporal: 3511
Orden cronológico: True


## 7. Comprobar la coherencia temporal

Comparamos la hora de corte obtenida de MT5 con la última vela M1 y el último tick

Esta comprobación es especialmente importante porque las velas
y los ticks deberán poder relacionarse posteriormente.

En `data/raw/` no modificaremos las marcas temporales originales.
Si aparece alguna discrepancia, se estudiará antes de transformar
los datos.

In [9]:
# --------------------------------------------------
# COMPROBAR COHERENCIA TEMPORAL
# --------------------------------------------------

ultima_m1 = m1_df["time"].max()
ultimo_tick = ticks_df["time"].max()


print(
    "Hora de corte MT5:",
    SNAPSHOT_TIME
)

print(
    "Última vela M1:",
    ultima_m1
)

print(
    "Último tick:",
    ultimo_tick
)


diferencia_m1_corte = (
    SNAPSHOT_TIME
    - ultima_m1
)

diferencia_tick_corte = (
    SNAPSHOT_TIME
    - ultimo_tick
)

diferencia_m1_tick = (
    ultima_m1
    - ultimo_tick
)


print(
    "\nDiferencia corte - última M1:",
    diferencia_m1_corte
)

print(
    "Diferencia corte - último tick:",
    diferencia_tick_corte
)

print(
    "Diferencia última M1 - último tick:",
    diferencia_m1_tick
)

Hora de corte MT5: 2026-09-08 00:17:00+00:00
Última vela M1: 2026-09-08 00:16:00+00:00
Último tick: 2026-09-08 00:16:56.458000+00:00

Diferencia corte - última M1: 0 days 00:01:00
Diferencia corte - último tick: 0 days 00:00:03.542000
Diferencia última M1 - último tick: -1 days +23:59:03.542000


## 8. Guardar la base histórica original

Guardamos las velas y los ticks con la fecha de la descarga.

Estos archivos constituirán nuestra fotografía histórica inicial
y no se sobrescribirán durante el desarrollo de la estrategia.

In [10]:
SNAPSHOT_DATE = tick_fin.strftime(
    "%Y%m%d"
)


M1_FILE = (
    DATA_RAW
    / f"eurusd_m1_{SNAPSHOT_DATE}.parquet"
)


TICKS_FILE = (
    DATA_RAW
    / f"eurusd_ticks_{SNAPSHOT_DATE}.parquet"
)


if M1_FILE.exists():
    raise FileExistsError(
        f"El archivo ya existe:\n{M1_FILE.resolve()}\n\n"
        "No se sobrescribe para proteger la base histórica."
    )


if TICKS_FILE.exists():
    raise FileExistsError(
        f"El archivo ya existe:\n{TICKS_FILE.resolve()}\n\n"
        "No se sobrescribe para proteger la base histórica."
    )


m1_df.to_parquet(
    M1_FILE,
    index=False
)


ticks_df.to_parquet(
    TICKS_FILE,
    index=False
)


print(
    "Archivo M1 guardado en:"
)

print(
    M1_FILE.resolve()
)


print(
    "\nArchivo de ticks guardado en:"
)

print(
    TICKS_FILE.resolve()
)

Archivo M1 guardado en:
C:\ALL\TECNICO\TRADING\CURSOS\TUTORIAS\PROYECTO2\mt5_microstructure_lab\data\raw\eurusd_m1_20260908.parquet

Archivo de ticks guardado en:
C:\ALL\TECNICO\TRADING\CURSOS\TUTORIAS\PROYECTO2\mt5_microstructure_lab\data\raw\eurusd_ticks_20260908.parquet


## 9. Resumen de la descarga

Mostramos la información principal de nuestra base histórica
antes de finalizar la conexión con MetaTrader 5.

In [11]:
print(
    "========== BASE HISTÓRICA =========="
)


print(
    "\nVELAS M1"
)

print(
    "Archivo:",
    M1_FILE.resolve()
)

print(
    "Número de velas:",
    len(m1_df)
)

print(
    "Desde:",
    m1_df["time"].min()
)

print(
    "Hasta:",
    m1_df["time"].max()
)


print(
    "\nTICKS BID/ASK"
)

print(
    "Archivo:",
    TICKS_FILE.resolve()
)

print(
    "Número de ticks:",
    len(ticks_df)
)

print(
    "Desde:",
    ticks_df["time"].min()
)

print(
    "Hasta:",
    ticks_df["time"].max()
)


print(
    "\nFecha de la fotografía:",
    SNAPSHOT_DATE
)

========== BASE HISTÓRICA ==========

VELAS M1
Archivo: C:\ALL\TECNICO\TRADING\CURSOS\TUTORIAS\PROYECTO2\mt5_microstructure_lab\data\raw\eurusd_m1_20260908.parquet
Número de velas: 99999
Desde: 2026-06-02 13:34:00+00:00
Hasta: 2026-09-08 00:16:00+00:00

TICKS BID/ASK
Archivo: C:\ALL\TECNICO\TRADING\CURSOS\TUTORIAS\PROYECTO2\mt5_microstructure_lab\data\raw\eurusd_ticks_20260908.parquet
Número de ticks: 1064413
Desde: 2026-09-01 00:17:05.574000+00:00
Hasta: 2026-09-08 00:16:56.458000+00:00

Fecha de la fotografía: 20260908


## 10. Cerrar la conexión con MetaTrader 5

Una vez descargada y guardada nuestra base histórica,
cerramos correctamente la conexión con MT5.

A partir de este momento podremos continuar el análisis en JupyterLab
sin necesidad de mantener MetaTrader 5 abierto.

In [12]:
mt5.shutdown()

print(
    "Conexión con MT5 cerrada correctamente."
)

Conexión con MT5 cerrada correctamente.
